In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# 1. Load data

In [ ]:
DATA_REAL = "../data/real/"
DATA_SYN = "../data/synthetic/"

distance_df = pd.read_csv(DATA_REAL + "distance_matrix.csv", index_col=0)
time_df = pd.read_csv(DATA_REAL + "time_matrix.csv", index_col=0)

cafes_df = pd.read_csv(DATA_SYN + "cafes.csv")
demands_df = pd.read_csv(DATA_SYN + "demands.csv")
products_df = pd.read_csv(DATA_SYN + "products.csv")
vans_df = pd.read_csv(DATA_SYN + "vans.csv")
depot_df = pd.read_csv(DATA_SYN + "depot.csv")
perish_df = pd.read_csv(DATA_SYN + "perishability_params.csv")

# 2. Sets

In [ ]:
depot_name = depot_df.loc[0, "name"]

cafes = cafes_df["cafe_name"].tolist()
nodes = [depot_name] + cafes

K = vans_df["van_id"].tolist()
P = products_df["product_id"].tolist()

milk_products = products_df.loc[
    products_df["category"] == "milk", "product_id"
].tolist()

A = [(i, j) for i in nodes for j in nodes if i != j]

# 3. Parameters

In [ ]:
cafe_id_to_name = dict(zip(cafes_df["cafe_id"], cafes_df["cafe_name"]))

demand_raw = {
    (row["cafe_id"], row["product_id"]): row["daily_demand"]
    for _, row in demands_df.iterrows()
}

d = {}
for cafe_id, cafe_name in cafe_id_to_name.items():
    for p in P:
        d[cafe_name, p] = demand_raw.get((cafe_id, p), 0)

w = dict(zip(products_df["product_id"], products_df["weight_per_unit_kg"]))
revenue = dict(zip(products_df["product_id"], products_df["revenue_per_unit"]))
cost = dict(zip(products_df["product_id"], products_df["cost_per_unit"]))
margin = {p: revenue[p] - cost[p] for p in P}

Q = dict(zip(vans_df["van_id"], vans_df["capacity_kg"]))
F = dict(zip(vans_df["van_id"], vans_df["fuel_cost_per_km"]))

D = {}
T = {}

for i, j in A:
    D[i, j] = float(distance_df.loc[i, j])
    T[i, j] = float(time_df.loc[i, j]) / 60.0

perish_params = dict(zip(perish_df["parameter"], perish_df["value"]))

T_max = float(perish_params["T_max_hours"])
loading_time = float(perish_params["loading_time_minutes"]) / 60.0
service_time = float(perish_params["service_time_minutes"]) / 60.0

beta = 5.0

M_milk = sum(d[i, p] for i in cafes for p in milk_products)
M_time = 1000

# 4. Helper: find subtours

In [ ]:

def find_subtours(selected_edges, cafes):
    """
    Find disconnected subtours among cafe nodes only.
    selected_edges: list of (i, j) arcs with x[i,j,k] = 1
    cafes: list of cafe nodes, excluding depot
    """
    unvisited = set(cafes)
    subtours = []

    # Build adjacency among cafes only
    successor = {}
    for i, j in selected_edges:
        if i in cafes and j in cafes:
            successor[i] = j

    while unvisited:
        start = next(iter(unvisited))
        tour = []
        current = start

        while current in unvisited:
            tour.append(current)
            unvisited.remove(current)

            if current not in successor:
                break

            current = successor[current]

        # A subtour is a closed cycle not connected to depot
        if len(tour) > 1 and current == start:
            subtours.append(tour)

    return subtours



# 5. DFJ Lazy Callback

In [ ]:

def dfj_callback(model, where):
    if where == GRB.Callback.MIPSOL:

        x = model._x
        cafes = model._cafes
        K = model._K

        for k in K:
            selected_edges = []

            for i in cafes:
                for j in cafes:
                    if i != j:
                        val = model.cbGetSolution(x[i, j, k])
                        if val > 0.5:
                            selected_edges.append((i, j))

            subtours = find_subtours(selected_edges, cafes)

            for S in subtours:
                model.cbLazy(
                    gp.quicksum(
                        x[i, j, k]
                        for i in S
                        for j in S
                        if i != j
                    )
                    <= len(S) - 1
                )


# 6. Build model

In [ ]:
m = gp.Model("profit_max_vrp_dfj_lazy")

x = m.addVars(A, K, vtype=GRB.BINARY, name="x")
y = m.addVars(cafes, K, vtype=GRB.BINARY, name="y")
z = m.addVars(cafes, vtype=GRB.BINARY, name="z")
q = m.addVars(cafes, P, K, lb=0, vtype=GRB.CONTINUOUS, name="q")
R = m.addVars(K, lb=0, vtype=GRB.CONTINUOUS, name="R")
g = m.addVars(K, vtype=GRB.BINARY, name="g")


# 7. Objective function

In [ ]:
product_margin = gp.quicksum(
    margin[p] * q[i, p, k]
    for i in cafes
    for p in P
    for k in K
)

transport_cost = gp.quicksum(
    F[k] * D[i, j] * x[i, j, k]
    for (i, j) in A
    for k in K
)

route_penalty = gp.quicksum(
    beta * R[k]
    for k in K
)

m.setObjective(
    product_margin - transport_cost - route_penalty,
    GRB.MAXIMIZE
)

# 8. Constraints

In [ ]:
# 1. Each cafe served by at most one van
m.addConstrs(
    (
        gp.quicksum(y[i, k] for k in K) == z[i]
        for i in cafes
    ),
    name="serve_once"
)

# 2. Demand satisfaction
m.addConstrs(
    (
        gp.quicksum(q[i, p, k] for k in K) == d[i, p] * z[i]
        for i in cafes
        for p in P
    ),
    name="demand_satisfaction"
)

# 3. Deliveries only if van services cafe
m.addConstrs(
    (
        q[i, p, k] <= d[i, p] * y[i, k]
        for i in cafes
        for p in P
        for k in K
    ),
    name="delivery_if_served"
)

# 4. Vehicle capacity
m.addConstrs(
    (
        gp.quicksum(
            w[p] * q[i, p, k]
            for i in cafes
            for p in P
        ) <= Q[k]
        for k in K
    ),
    name="capacity"
)

# 5. Flow conservation
m.addConstrs(
    (
        gp.quicksum(x[i, j, k] for j in nodes if j != i) == y[i, k]
        for i in cafes
        for k in K
    ),
    name="flow_out"
)

m.addConstrs(
    (
        gp.quicksum(x[j, i, k] for j in nodes if j != i) == y[i, k]
        for i in cafes
        for k in K
    ),
    name="flow_in"
)

# 6. Depot departure and return
m.addConstrs(
    (
        gp.quicksum(x[depot_name, j, k] for j in cafes) <= 1
        for k in K
    ),
    name="depot_departure"
)

m.addConstrs(
    (
        gp.quicksum(x[i, depot_name, k] for i in cafes) <= 1
        for k in K
    ),
    name="depot_return"
)

m.addConstrs(
    (
        gp.quicksum(x[depot_name, j, k] for j in cafes)
        ==
        gp.quicksum(x[i, depot_name, k] for i in cafes)
        for k in K
    ),
    name="depot_balance"
)

# 7. Route duration
m.addConstrs(
    (
        R[k]
        ==
        loading_time * gp.quicksum(x[depot_name, j, k] for j in cafes)
        +
        gp.quicksum(T[i, j] * x[i, j, k] for (i, j) in A)
        +
        service_time * gp.quicksum(y[i, k] for i in cafes)
        for k in K
    ),
    name="route_duration"
)

# 8. Milk indicator
m.addConstrs(
    (
        gp.quicksum(
            q[i, p, k]
            for i in cafes
            for p in milk_products
        )
        <= M_milk * g[k]
        for k in K
    ),
    name="milk_indicator"
)

# 9. Maximum route duration if carrying milk
m.addConstrs(
    (
        R[k] <= T_max + M_time * (1 - g[k])
        for k in K
    ),
    name="milk_time_limit"
)


# 9. Attach callback data

In [ ]:
m._x = x
m._cafes = cafes
m._K = K

m.Params.LazyConstraints = 1
m.Params.OutputFlag = 1
m.Params.TimeLimit = 300

# 10. Solve

In [ ]:
m.optimize(dfj_callback)

# 11. Print solution

In [ ]:
if m.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:

    print("\nObjective value:", m.ObjVal)

    print("\nServed cafes:")
    for i in cafes:
        if z[i].X > 0.5:
            print(f"- {i}")

    print("\nVan routes:")

    for k in K:
        used = sum(x[depot_name, j, k].X for j in cafes)

        if used > 0.5:
            print(f"\n{k}")
            print(f"Route duration: {R[k].X:.2f} hours")
            print(f"Carries milk: {round(g[k].X)}")

            route = [depot_name]
            current = depot_name

            while True:
                next_nodes = [
                    j for j in nodes
                    if j != current and x[current, j, k].X > 0.5
                ]

                if not next_nodes:
                    break

                next_node = next_nodes[0]
                route.append(next_node)

                if next_node == depot_name:
                    break

                current = next_node

            print(" -> ".join(route))

            total_weight = sum(
                w[p] * q[i, p, k].X
                for i in cafes
                for p in P
            )

            print(f"Load: {total_weight:.2f} kg / {Q[k]} kg")

else:
    print("Model was not solved successfully.")
    print("Status:", m.status)